# **Orchestral Agent Setup Basics**

**Written by**: 
- Tony Menzo (U. of Alabama / Fermilab)
- Alex Roman (U. of Alabama)

This notebook demonstrates how to set up and interact with AI agents using the **Orchestral** orchestration engine. It serves as the foundational guide for building agentic workflows.

**What you'll learn:**
1. Configuring LLM providers (GPT, Claude, Gemini, Groq, Ollama)
2. Creating and configuring Orchestral agents
3. Registering tools and defining system prompts
4. Interacting with agents programmatically and via web UI
5. Building custom tools

---

[**Orchestral**](https://orchestral-ai.com) is a Python framework for building AI agents with tool-use capabilities. It provides a unified interface across multiple LLM providers and a flexible tool system.

**Orchestral AI Repository**: [https://github.com/orchestralAI/orchestral-ai](https://github.com/orchestralAI/orchestral-ai)

**For HEPTAPOD physics tools** (FeynRules, MadGraph, Pythia, etc.), see `heptapod_setup.ipynb` for configuration of external tools, `config.py`, and the test runner.

## 0. Prerequisites

### **Environment Setup**

We recommend using a fresh Python environment:

```bash
# Using conda
conda create -n heptapod python=3.13
conda activate heptapod

# Or using venv
python -m venv heptapod-env
source heptapod-env/bin/activate  # Linux/macOS
```

### **Installation**

```bash
# Install all HEPTAPOD dependencies (includes Orchestral and physics tools)
pip install -r requirements.txt

# Or install just Orchestral and LLM providers
pip install orchestral-ai openai anthropic google-generativeai
```

### **LLM API Keys**

Orchestral supports multiple LLM providers. Create a `.env` file in the repository root with your API keys:

```bash
# .env file - add the providers you want to use
OPENAI_API_KEY=sk-...           # For GPT models
ANTHROPIC_API_KEY=sk-ant-...    # For Claude models
GOOGLE_API_KEY=...              # For Gemini models
GROQ_API_KEY=gsk_...            # For Groq (fast inference)
MISTRAL_API_KEY=...             # For Mistral models
```

**Note:** You only need API keys for the providers you plan to use. For local models, install [Ollama](https://ollama.ai) instead.

**Quick Setup:** If you've already completed `heptapod_setup.ipynb`, you can use the shared setup module:

```python
from examples.shared.heptapod_setup import setup_heptapod
config = setup_heptapod()  # Displays config and returns dict
```

In [ ]:
# ============================================================================
# SETUP CELL
# ============================================================================
# Run this once per session. Choose ONE of the two options below.

import sys
from pathlib import Path

# Add repository root to path (examples/orchestral/ -> 2 levels up)
REPO_ROOT = Path.cwd().parent.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"Repository root: {REPO_ROOT}")

# -----------------------------------------------------------------------------
# OPTION A: Quick setup using shared module (if you completed heptapod_setup.ipynb)
# -----------------------------------------------------------------------------
# from examples.shared.heptapod_setup import setup_heptapod
# config = setup_heptapod()  # Displays full config status

# -----------------------------------------------------------------------------
# OPTION B: Manual setup (default - runs below)
# -----------------------------------------------------------------------------
from dotenv import load_dotenv
import os

env_path = REPO_ROOT / ".env"
if env_path.exists():
    load_dotenv(env_path)
    print(f"Loaded .env from {env_path}")
    
    # Show which providers are available
    providers = [
        ("OPENAI_API_KEY", "OpenAI"),
        ("ANTHROPIC_API_KEY", "Anthropic"),
        ("GOOGLE_API_KEY", "Google"),
        ("GROQ_API_KEY", "Groq"),
    ]
    available = [name for key, name in providers if os.getenv(key)]
    print(f"Available providers: {', '.join(available) if available else 'None configured'}")
else:
    print(f"No .env file found at {env_path}")

print("\nSetup complete!")

In [ ]:
# Core Orchestral imports
from orchestral import Agent
from orchestral.tools import (
    RunCommandTool,
    WriteFileTool,
    ReadFileTool,
    EditFileTool,
    FileSearchTool,
    FindFilesTool,
    RunPythonTool,
    TodoWrite,
    TodoRead
)
from orchestral.tools.hooks import TruncateOutputHook

# LLM provider imports
from orchestral.llm import GPT, Claude, Gemini, Groq
from llm import get_ollama

print("Imports successful!")

## 1. Configure Workspace

Set up a working directory for the agent to store files and outputs.

In [ ]:
# Create a sandbox directory for this demo - use local sandbox_notebook
demo_dir = Path.cwd() / 'sandbox_notebook'
demo_dir.mkdir(exist_ok=True)

base_directory = str(demo_dir)
print(f"Working directory: {base_directory}")

## 2. Choose Your LLM Provider

**Option A: OpenAI GPT**
- Requires `OPENAI_API_KEY` in `.env` file
- Good reasoning and physics knowledge

**Option B: Anthropic Claude**
- Requires `ANTHROPIC_API_KEY` in `.env`
- Excellent reasoning and long context

**Option C: Google Gemini**
- Requires `GOOGLE_API_KEY` in `.env`
- Very long context windows (up to 2M tokens)

**Option D: Groq**
- Requires `GROQ_API_KEY` in `.env`
- Extremely fast inference (specialized hardware)

**Option E: Local Ollama**
- Free, runs locally
- Configure model in `config.py`

### 2.1 List Available Models

Run this cell to see all available models for each provider.

In [ ]:
from examples.shared.llm_utils import list_available_models

models = list_available_models()

In [ ]:
# CHOOSE ONE:

# Option 1: OpenAI GPT (default model)
llm = GPT()

# Option 1b: OpenAI GPT with specific model
# llm = GPT(model='gpt-4o-mini')

# Option 2: Anthropic Claude (default model)
# llm = Claude()

# Option 2b: Claude with specific model
# llm = Claude(model='claude-sonnet-4-5-20250929')

# Option 3: Google Gemini (default model)
# llm = Gemini()

# Option 3b: Gemini with specific model
# llm = Gemini(model='gemini-2.0-flash-exp')

# Option 4: Groq (fast inference, default model)
# llm = Groq()

# Option 4b: Groq with specific model
# llm = Groq(model='llama-3.3-70b-versatile')

# Option 5: Local Ollama (uses config.py settings)
# llm = get_ollama()

# Option 5b: Ollama with specific model
# llm = get_ollama(model='llama3:70b')

print(f"Using LLM: {llm.__class__.__name__}")

## 3. Define Tools

Tools are capabilities the agent can use. Start with basic file/command tools.

In [ ]:
tools = [
    # File operations
    WriteFileTool(base_directory=base_directory),
    ReadFileTool(base_directory=base_directory, show_line_numbers=True),
    EditFileTool(base_directory=base_directory),
    FindFilesTool(base_directory=base_directory),
    FileSearchTool(base_directory=base_directory),
    
    # Command execution
    RunCommandTool(base_directory=base_directory),
    RunPythonTool(base_directory=base_directory, timeout=1000),
    
    # Task management
    TodoRead(),
    TodoWrite(base_directory=base_directory)
]

# Optional: Add hooks to control tool behavior
hooks = [
    TruncateOutputHook(max_length=10000),  # Prevent huge outputs
]

print(f"Configured {len(tools)} tools")

## 4. Define System Prompt

The system prompt guides the agent's behavior and expertise.

In [ ]:
system_prompt = """
You are a helpful physics research assistant with expertise in particle physics.

Your capabilities:
- File operations (read, write, edit, search)
- Running Python code and shell commands
- Task management

When solving problems:
1. Break down complex tasks into steps
2. Explain your reasoning clearly
3. Write clean, well-documented code
4. Verify your results

Be concise but thorough in your responses.
""".strip()

print("System prompt defined")

## 5. Create the Agent

Combine LLM, tools, hooks, and prompt into an agent instance.

In [ ]:
agent = Agent(
    llm=llm,
    tools=tools,
    tool_hooks=hooks,
    system_prompt=system_prompt,
    debug=False  # Set True to see detailed execution logs
)

print("Agent created successfully!")

## 6. Interact with Agent (Programmatic)

You can call the agent directly from Python code.

In [ ]:
# Simple query
response = agent.run("What is the speed of light in natural units?")
print(response)

In [ ]:
# Task involving tool use
response = agent.run(
    "Create a Python file called 'compton.py' that calculates "
    "the Compton wavelength of an electron."
)
print(response)

In [ ]:
# Verify the file was created
test_file = demo_dir / 'compton.py'
if test_file.exists():
    print("File created!\n")
    print(test_file.read_text())
else:
    print("File not found")

In [ ]:
# Run the generated Python file and show its output
import subprocess

test_file = demo_dir / 'compton.py'
if test_file.exists():
    result = subprocess.run(
        [sys.executable, str(test_file)],
        capture_output=True,
        text=True,
        cwd=demo_dir
    )
    print("Output:")
    print("-" * 40)
    if result.stdout:
        print(result.stdout)
    if result.stderr:
        print("Errors:", result.stderr)
else:
    print("File not found - run the previous cell first")

## 7. Interact with Agent (Web Interface)

For longer conversations, use the web interface.

In [ ]:
import app.server as app_server

# This will:
# 1. Start a local web server
# 2. Open your browser
# 3. Provide a chat interface to the agent

# UNCOMMENT to start web server:
# app_server.run_server(
#     agent,
#     host="127.0.0.1",
#     port=8000,
#     open_browser=True,
#     max_tool_iterations=50
# )

print("Note: Uncomment the code above to start the web interface.")
print("The server will run until you stop it (Kernel > Interrupt).")

## 8. Adding Custom Tools

Orchestral makes it easy to create custom tools. We'll demonstrate two patterns:

1. **`@define_tool` decorator** - Turn any Python function into a tool with one line
2. **Stateful tools** - Tools that maintain state or access external resources (files, databases, APIs)

We'll also use a **ToolCallLogger** hook to see exactly which tools the agent calls and with what arguments - useful for debugging and understanding tool chaining.

For production use, HEPTAPOD provides full-featured tools in `tools/units/`, `tools/nda/`, `tools/feyngraph/`, etc.

In [ ]:
# Import the ToolCallLogger to see tool calls in action
from examples.shared.tool_logger import ToolCallLogger

# Configure logging - toggle these to control verbosity
ENABLE_TOOL_LOGGING = True  # Set to False to disable logging

if ENABLE_TOOL_LOGGING:
    tool_logger = ToolCallLogger(
        verbose=True,       # Show full arguments (False = just tool names)
        show_results=False  # Show tool outputs (can be very verbose)
    )
    # Add to our hooks list
    logging_hooks = hooks + [tool_logger]
    print("Tool call logging ENABLED")
    print("  You'll see >>> TOOL CALL and <<< completed messages")
else:
    logging_hooks = hooks
    print("Tool call logging DISABLED")

### 8.1 Simple Tools with `@define_tool`

The fastest way to create a tool is with the `@define_tool` decorator. Just write a normal Python function with type hints and a docstring - Orchestral handles the rest.

In [ ]:
from orchestral.tools import define_tool

# =============================================================================
# Special Relativity Tools
# =============================================================================

@define_tool()
def lorentz_gamma(velocity: float) -> str:
    """
    Calculate the Lorentz factor (gamma) for a given velocity.
    
    IMPORTANT: Call this tool FIRST to get gamma, then use that value 
    with the time_dilation tool.
    
    Args:
        velocity: Velocity as a fraction of the speed of light (0 < v < 1).
                  For example, 0.5 means half the speed of light.
    
    Returns:
        The Lorentz factor gamma = 1 / sqrt(1 - v^2).
    """
    import math
    
    if velocity <= 0 or velocity >= 1:
        return f"Error: velocity must be between 0 and 1 (got {velocity})"
    
    gamma = 1 / math.sqrt(1 - velocity**2)
    
    return (
        f"For v = {velocity}c:\n"
        f"  Lorentz factor: gamma = {gamma:.6f}\n"
        f"\nUse this gamma value with the time_dilation tool."
    )


@define_tool()
def time_dilation(proper_time: float, gamma: float, velocity: float) -> str:
    """
    Calculate the dilated time observed in the lab frame.
    
    Use this to find how long a moving particle lives in the lab frame,
    given its proper lifetime (rest frame lifetime) and the Lorentz factor.
    
    IMPORTANT: You must first call lorentz_gamma to get the gamma value!
    
    Args:
        proper_time: The proper time (rest frame time) in microseconds.
                     For example, muon proper lifetime is 2.2 microseconds.
        gamma: The Lorentz factor (get this from lorentz_gamma tool first).
        velocity: Velocity as a fraction of the speed of light (0 < v < 1).
    
    Returns:
        The dilated time in microseconds and the distance traveled.
    """
    if gamma < 1:
        return f"Error: gamma must be >= 1 (got {gamma}). Did you call lorentz_gamma first?"
    
    dilated_time = proper_time * gamma
    
    # Calculate distance traveled (in meters, using c = 3e8 m/s)
    c = 299792458  # m/s
    distance = velocity * c * dilated_time * 1e-6  # convert microseconds to seconds
    
    return (
        f"Time dilation calculation:\n"
        f"  Proper time (rest frame): {proper_time:.3f} microseconds\n"
        f"  Lorentz factor: gamma = {gamma:.4f}\n"
        f"  Dilated time (lab frame): {dilated_time:.3f} microseconds\n"
        f"  At v = {velocity}c, distance traveled: {distance:.1f} m ({distance/1000:.2f} km)"
    )


# Exercise: Implement length_contraction(proper_length, gamma) following the same pattern!


print("Created 2 special relativity tools:")
print(f"  1. lorentz_gamma")
print(f"  2. time_dilation - requires gamma from lorentz_gamma")
print("\nExercise: Implement a length_contraction tool following the same pattern!")

In [ ]:
# Create an agent with the special relativity tools (with logging!)
relativity_tools = [lorentz_gamma, time_dilation]

agent_relativity = Agent(
    llm=llm,
    tools=tools + relativity_tools,
    tool_hooks=logging_hooks,  # Use logging_hooks to see tool calls
    system_prompt=system_prompt,
    debug=False
)

# Ask about muon lifetime - agent must use lorentz_gamma first, then time_dilation
# Watch the >>> TOOL CALL messages to see the chaining!
response = agent_relativity.run(
    "A muon has a proper lifetime of 2.2 microseconds. If it travels at 0.995c, "
    "how long will it live in the lab frame and how far will it travel? "
    "First calculate gamma using lorentz_gamma, then use that result with time_dilation."
)
print(response)

In [ ]:
# =============================================================================
# Uncertainty Propagation Tools
# =============================================================================
# These tools are composable - results from one can feed into another.
# The agent chains them together to propagate uncertainties through calculations.

import math
from typing import List

@define_tool()
def value_with_uncertainty(value: float, uncertainty: float) -> str:
    """
    Create a measurement with its uncertainty. Use this as a starting point
    for uncertainty propagation calculations.
    
    Args:
        value: The measured value.
        uncertainty: The absolute uncertainty (standard deviation).
    
    Returns:
        Formatted measurement that can be used with other uncertainty tools.
    """
    rel_unc = abs(uncertainty / value) * 100 if value != 0 else float('inf')
    return (
        f"Measurement:\n"
        f"  Value: {value}\n"
        f"  Uncertainty: +/- {uncertainty}\n"
        f"  Relative uncertainty: {rel_unc:.2f}%\n"
        f"  Result: {value} +/- {uncertainty}"
    )


@define_tool()
def add_measurements(a: float, da: float, b: float, db: float) -> str:
    """
    Add two measurements with uncertainty propagation.
    For addition/subtraction: uncertainties add in quadrature.
    
    Formula: δ(a+b) = sqrt(δa² + δb²)
    
    Args:
        a: First value.
        da: Uncertainty in first value.
        b: Second value.
        db: Uncertainty in second value.
    
    Returns:
        Sum with propagated uncertainty.
    """
    result = a + b
    uncertainty = math.sqrt(da**2 + db**2)
    rel_unc = abs(uncertainty / result) * 100 if result != 0 else float('inf')
    
    return (
        f"Addition with uncertainty propagation:\n"
        f"  ({a} +/- {da}) + ({b} +/- {db})\n"
        f"  Result: {result} +/- {uncertainty:.6g}\n"
        f"  Relative uncertainty: {rel_unc:.2f}%"
    )


@define_tool()
def subtract_measurements(a: float, da: float, b: float, db: float) -> str:
    """
    Subtract two measurements with uncertainty propagation.
    For addition/subtraction: uncertainties add in quadrature.
    
    Formula: δ(a-b) = sqrt(δa² + δb²)
    
    Args:
        a: First value.
        da: Uncertainty in first value.
        b: Second value.
        db: Uncertainty in second value.
    
    Returns:
        Difference with propagated uncertainty.
    """
    result = a - b
    uncertainty = math.sqrt(da**2 + db**2)
    rel_unc = abs(uncertainty / result) * 100 if result != 0 else float('inf')
    
    return (
        f"Subtraction with uncertainty propagation:\n"
        f"  ({a} +/- {da}) - ({b} +/- {db})\n"
        f"  Result: {result} +/- {uncertainty:.6g}\n"
        f"  Relative uncertainty: {rel_unc:.2f}%"
    )


@define_tool()
def multiply_measurements(a: float, da: float, b: float, db: float) -> str:
    """
    Multiply two measurements with uncertainty propagation.
    For multiplication/division: relative uncertainties add in quadrature.
    
    Formula: δ(a*b)/(a*b) = sqrt((δa/a)² + (δb/b)²)
    
    Args:
        a: First value.
        da: Uncertainty in first value.
        b: Second value.
        db: Uncertainty in second value.
    
    Returns:
        Product with propagated uncertainty.
    """
    result = a * b
    
    # Relative uncertainties
    rel_a = da / abs(a) if a != 0 else 0
    rel_b = db / abs(b) if b != 0 else 0
    rel_result = math.sqrt(rel_a**2 + rel_b**2)
    uncertainty = abs(result) * rel_result
    
    return (
        f"Multiplication with uncertainty propagation:\n"
        f"  ({a} +/- {da}) * ({b} +/- {db})\n"
        f"  Relative uncertainties: {rel_a*100:.2f}% and {rel_b*100:.2f}%\n"
        f"  Result: {result} +/- {uncertainty:.6g}\n"
        f"  Relative uncertainty: {rel_result*100:.2f}%"
    )


@define_tool()
def divide_measurements(a: float, da: float, b: float, db: float) -> str:
    """
    Divide two measurements with uncertainty propagation.
    For multiplication/division: relative uncertainties add in quadrature.
    
    Formula: δ(a/b)/(a/b) = sqrt((δa/a)² + (δb/b)²)
    
    Args:
        a: Numerator value.
        da: Uncertainty in numerator.
        b: Denominator value.
        db: Uncertainty in denominator.
    
    Returns:
        Quotient with propagated uncertainty.
    """
    if b == 0:
        return "Error: Cannot divide by zero."
    
    result = a / b
    
    # Relative uncertainties
    rel_a = da / abs(a) if a != 0 else 0
    rel_b = db / abs(b)
    rel_result = math.sqrt(rel_a**2 + rel_b**2)
    uncertainty = abs(result) * rel_result
    
    return (
        f"Division with uncertainty propagation:\n"
        f"  ({a} +/- {da}) / ({b} +/- {db})\n"
        f"  Relative uncertainties: {rel_a*100:.2f}% and {rel_b*100:.2f}%\n"
        f"  Result: {result} +/- {uncertainty:.6g}\n"
        f"  Relative uncertainty: {rel_result*100:.2f}%"
    )


@define_tool()
def power_measurement(a: float, da: float, n: float) -> str:
    """
    Raise a measurement to a power with uncertainty propagation.
    
    Formula: δ(a^n) = |n| * a^(n-1) * δa = |n| * (a^n) * (δa/a)
    
    Args:
        a: Base value.
        da: Uncertainty in base value.
        n: Exponent (can be fractional, e.g., 0.5 for square root).
    
    Returns:
        Result with propagated uncertainty.
    """
    if a <= 0 and n != int(n):
        return f"Error: Cannot raise non-positive number {a} to fractional power {n}."
    
    result = a ** n
    
    # For a^n: relative uncertainty multiplied by |n|
    rel_a = da / abs(a) if a != 0 else 0
    rel_result = abs(n) * rel_a
    uncertainty = abs(result) * rel_result
    
    operation = f"√" if n == 0.5 else f"^{n}"
    
    return (
        f"Power with uncertainty propagation:\n"
        f"  ({a} +/- {da}){operation}\n"
        f"  Relative uncertainty scales by |n|={abs(n)}: {rel_a*100:.2f}% → {rel_result*100:.2f}%\n"
        f"  Result: {result:.6g} +/- {uncertainty:.6g}\n"
        f"  Relative uncertainty: {rel_result*100:.2f}%"
    )


@define_tool()
def combine_uncertainties(uncertainties: List[float]) -> str:
    """
    Combine multiple independent uncertainties in quadrature.
    
    Formula: δ_total = sqrt(δ1² + δ2² + δ3² + ...)
    
    This is useful for combining systematic uncertainties from different sources.
    
    Args:
        uncertainties: List of uncertainty values to combine, e.g., [0.1, 0.2, 0.15].
    
    Returns:
        Combined uncertainty.
    """
    if not uncertainties:
        return "Error: No uncertainties provided."
    
    combined = math.sqrt(sum(u**2 for u in uncertainties))
    
    terms = " + ".join([f"({u})²" for u in uncertainties])
    
    return (
        f"Combining {len(uncertainties)} uncertainties in quadrature:\n"
        f"  \\sqrt({terms})\n"
        f"  = \\sqrt({sum(u**2 for u in uncertainties):.6g})\n"
        f"  = {combined:.6g}"
    )


# List all uncertainty tools
uncertainty_tools = [
    value_with_uncertainty,
    add_measurements,
    subtract_measurements, 
    multiply_measurements,
    divide_measurements,
    power_measurement,
    combine_uncertainties,
]

print(f"Created {len(uncertainty_tools)} uncertainty propagation tools:")
print("  - value_with_uncertainty")
print("  - add_measurements")
print("  - subtract_measurements")
print("  - multiply_measurements")
print("  - divide_measurements")
print("  - power_measurement")
print("  - combine_uncertainties")

In [ ]:
# Create an agent with uncertainty propagation tools (with logging!)
agent_uncertainty = Agent(
    llm=llm,
    tools=tools + uncertainty_tools,
    tool_hooks=logging_hooks,  # Use logging_hooks to see tool chaining
    system_prompt=system_prompt,
    debug=False
)

# Example: Calculate kinetic energy E = (1/2)mv^2 with uncertainties (the correct uncertainty is \pm 13.5 J)
# This requires the agent to chain multiple tools - watch the >>> TOOL CALL messages!
response = agent_uncertainty.run(
    "A particle has mass m = 2.5 +/- 0.1 kg and velocity v = 10.0 +/- 0.5 m/s. "
    "Calculate the kinetic energy E = (1/2)mv^2 with proper uncertainty propagation. "
    "First square the velocity using power_measurement, then multiply by mass, then by 0.5."
)
print(response)

### 8.2 Stateful Tools with External Data

For tools that need to access files, databases, or maintain state, subclass `BaseTool`. This example creates a particle lookup tool that reads from a JSON database file.

In [ ]:
# First, create a JSON database file in our sandbox
import json

particle_database = {
    "electron": {"mass_mev": 0.511, "charge": -1, "spin": 0.5, "type": "lepton"},
    "muon": {"mass_mev": 105.7, "charge": -1, "spin": 0.5, "type": "lepton"},
    "tau": {"mass_mev": 1776.9, "charge": -1, "spin": 0.5, "type": "lepton"},
    "proton": {"mass_mev": 938.3, "charge": 1, "spin": 0.5, "type": "baryon"},
    "neutron": {"mass_mev": 939.6, "charge": 0, "spin": 0.5, "type": "baryon"},
    "higgs": {"mass_mev": 125100, "charge": 0, "spin": 0, "type": "boson"},
    "w_plus": {"mass_mev": 80400, "charge": 1, "spin": 1, "type": "boson"},
    "w_minus": {"mass_mev": 80400, "charge": -1, "spin": 1, "type": "boson"},
    "z": {"mass_mev": 91200, "charge": 0, "spin": 1, "type": "boson"},
    "photon": {"mass_mev": 0, "charge": 0, "spin": 1, "type": "boson"},
    "top": {"mass_mev": 172760, "charge": 0.667, "spin": 0.5, "type": "quark"},
    "bottom": {"mass_mev": 4180, "charge": -0.333, "spin": 0.5, "type": "quark"},
}

db_path = demo_dir / "particles.json"
db_path.write_text(json.dumps(particle_database, indent=2))
print(f"Created particle database: {db_path}")
print(f"Contains {len(particle_database)} particles")

In [ ]:
from orchestral.tools import BaseTool
from orchestral.tools.base.field_utils import RuntimeField, StateField
import json

class ParticleDatabaseTool(BaseTool):
    """A stateful tool that looks up particle properties from a JSON database file."""
    
    # ======================== Runtime fields ======================== #
    particle_name: str = RuntimeField(
        description="Particle name to look up (e.g., 'electron', 'muon', 'higgs')"
    )
    # ================================================================ #
    
    # ========================= State fields ========================= #
    base_directory: str = StateField(
        description="Base sandbox directory for file operations"
    )
    database_path: str = StateField(
        description="Path to the JSON particle database file"
    )
    # ================================================================ #
    
    def _setup(self):
        """Initialize the tool and load the database."""
        self._db_path = Path(self.database_path)
        self._load_database()
    
    def _load_database(self):
        """Load particle data from JSON file."""
        if self._db_path.exists():
            self._particles = json.loads(self._db_path.read_text())
        else:
            self._particles = {}
            print(f"Warning: Database not found at {self._db_path}")
    
    def _run(self) -> str:
        """Look up a particle in the database."""
        # Reload database each call to pick up any changes
        self._load_database()
        
        name = self.particle_name.lower().strip()
        
        if name in self._particles:
            p = self._particles[name]
            mass = p['mass_mev']
            # Format mass nicely
            if mass >= 1000:
                mass_str = f"{mass/1000:.2f} GeV"
            else:
                mass_str = f"{mass:.3f} MeV"
            
            charge_str = f"{p['charge']:+.3f}e" if p['charge'] != 0 else "0"
            
            return (
                f"{self.particle_name}:\n"
                f"  Mass: {mass_str}\n"
                f"  Charge: {charge_str}\n"
                f"  Spin: {p['spin']}\n"
                f"  Type: {p['type']}"
            )
        else:
            available = ', '.join(sorted(self._particles.keys()))
            return f"Particle '{self.particle_name}' not found.\nAvailable: {available}"


# Create the tool, pointing to our database file
particle_tool = ParticleDatabaseTool(
    database_path=str(db_path),
    base_directory=base_directory
)

print(f"ParticleDatabaseTool created!")
print(f"  Database: {db_path}")
print(f"  Available particles: {', '.join(sorted(particle_tool._particles.keys()))}")

In [ ]:
# Add the custom tool to our existing tools list
enhanced_tools = tools + [particle_tool]

# Reinitialize the agent with the enhanced tool set (with logging!)
agent_enhanced = Agent(
    llm=llm,
    tools=enhanced_tools,
    tool_hooks=logging_hooks,  # Use logging_hooks to see tool calls
    system_prompt=system_prompt,
    debug=False
)

print(f"Agent reinitialized with ParticleDatabaseTool!")
print(f"Total tools: {len(enhanced_tools)}")

In [ ]:
# Test the enhanced agent with the database-backed tool
response = agent_enhanced.run(
    "Compare the masses of the top quark and the Higgs boson. Use the ParticleDatabase tool."
)
print(response)

## Summary

This notebook showed the complete pattern for setting up Orchestral agents:

1. **Setup**: Import libraries and configure paths
2. **LLM**: Choose provider (GPT, Claude, Gemini, Groq, Ollama)
3. **Model Discovery**: Use `get_available_models()` to explore options
4. **Tools**: Register capabilities for the agent
5. **System Prompt**: Define expertise and behavior
6. **Agent**: Combine components
7. **Interact**: Use programmatically or via web UI
8. **Extend**: Create custom tools and reinitialize the agent

**Next steps - Production Tools:**

*Simple but powerful tools (no external dependencies):*
- See `examples/units/` for **NaturalUnitsConverter** - convert between natural and SI units
- See `examples/pdg/` for **PDGDatabaseTool** - query official particle properties
- See `examples/inspire/` for **InspireTools** - search HEP literature

*Event generation pipeline (requires external software):*
- See `examples/feynrules/` for **FeynRules** BSM model creation
- See `examples/madgraph/` for **MadGraph** parton-level event generation
- See `examples/event_generation/` for **Pythia** and **Sherpa** showering/hadronization

*Complete workflows:*
- See `examples/workflows/bsm/hep_bsm_tutorial.ipynb` for a full BSM analysis pipeline

## Cleanup (Optional)

In [ ]:
# Uncomment to remove the demo sandbox directory
# import shutil
# if demo_dir.exists():
#     shutil.rmtree(demo_dir)
#     print("✓ Cleaned up demo directory")

print("Demo complete!")